# VoiceSecure - CosyVoice3 기반 학습 (Kaggle)

**필수 설정 (우상단 Settings):**
- Accelerator: **GPU T4 x2** (또는 P100)
- Internet: **ON**
- Dataset: **bryanpark/korean-single-speaker-speech-dataset** (사이드바 Add Data)

**🚨 학습 셀(셀 7) 전 또는 직후 반드시 `Save Version → Save & Run All (Commit)` 누르세요.**
그래야 세션 끊겨도 결과 보존됩니다.

**실행 순서:** 셀 1~3 → Restart Session → 셀 4~7 → (Save & Run All) → 셀 8 평가

## 셀 1 — 환경 확인

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "❌ NONE"}')
print(f'torch: {torch.__version__}')

## 셀 2 — Repo clone (SDK + CosyVoice)

In [ ]:
import os, sys

KAGGLE_WORKING = '/kaggle/working'
SDK_ROOT       = f'{KAGGLE_WORKING}/voicesecure-sdk'
COSY_ROOT      = f'{KAGGLE_WORKING}/CosyVoice'

# 1) voicesecure-sdk (cosyvoice 통합 branch)
if not os.path.exists(SDK_ROOT):
    !git clone -b feat/xtts-self-encoder-reward https://github.com/VoiceSecureHoseo/voicesecure-sdk.git {SDK_ROOT}
else:
    !cd {SDK_ROOT} && git pull

# 2) CosyVoice repo (Matcha-TTS submodule 포함)
if not os.path.exists(COSY_ROOT):
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSY_ROOT}

# 3) sys.path 등록
for p in [SDK_ROOT, f'{SDK_ROOT}/src', COSY_ROOT, f'{COSY_ROOT}/third_party/Matcha-TTS']:
    if p not in sys.path:
        sys.path.insert(0, p)

print('✓ Repo clone 완료')

## 셀 3 — 패키지 설치 (실행 후 Restart Session 필수)

In [ ]:
# SDK 의존성
%cd {SDK_ROOT}
!pip install -q -e .

# CosyVoice 핵심 의존성 (frontend 가 whisper, omegaconf, inflect 등도 import)
!pip install -q hyperpyyaml conformer modelscope huggingface_hub
!pip install -q lightning pytorch-lightning gdown librosa onnxruntime-gpu
!pip install -q diffusers wetext WeTextProcessing
!pip install -q openai-whisper inflect omegaconf pydantic
!pip install -q tensorboard tensorboardx rich gradio fastapi protobuf grpcio
!pip install -q einops wget

# Matcha-TTS requirements 통째 (CosyVoice flow_matching 의존성 전체)
!pip install -q -r {COSY_ROOT}/third_party/Matcha-TTS/requirements.txt

# CosyVoice requirements 통째 (Matcha 가 못 잡는 나머지)
!pip install -q -r {COSY_ROOT}/requirements.txt 2>&1 | tail -10

# transformers 호환 버전 강제 (CosyVoice3 Qwen2 호환)
!pip install -q "transformers>=4.40,<4.50" "tokenizers>=0.19"

# SpeechBrain (ECAPA-TDNN 평가용 — 평가 시 사용)
!pip install -q speechbrain

print('✓ 설치 완료 — Run → Restart Session 누르세요')

---
## ⚠️ 여기서 Run → Restart Session 누르고 셀 4부터 계속
---

## 셀 4 — 재시작 후 sys.path 복구 + CosyVoice3 모델 다운

In [ ]:
import os, sys

KAGGLE_WORKING = '/kaggle/working'
SDK_ROOT       = f'{KAGGLE_WORKING}/voicesecure-sdk'
COSY_ROOT      = f'{KAGGLE_WORKING}/CosyVoice'
COSY_MODEL_DIR = f'{COSY_ROOT}/pretrained_models/Fun-CosyVoice3-0.5B-2512'

for p in [SDK_ROOT, f'{SDK_ROOT}/src', COSY_ROOT, f'{COSY_ROOT}/third_party/Matcha-TTS']:
    if p not in sys.path:
        sys.path.insert(0, p)

# CosyVoice3 모델 다운 (HuggingFace)
if not os.path.exists(COSY_MODEL_DIR) or not os.listdir(COSY_MODEL_DIR):
    from huggingface_hub import snapshot_download
    os.makedirs(COSY_MODEL_DIR, exist_ok=True)
    print('CosyVoice3 모델 다운 중 (~2GB, 5~10분)...')
    snapshot_download(
        repo_id='FunAudioLLM/Fun-CosyVoice3-0.5B-2512',
        local_dir=COSY_MODEL_DIR,
    )

print('✓ 모델 다운 완료')
!ls -lh {COSY_MODEL_DIR}/ | head -20

## 셀 5 — CosyVoice3 어댑터 로드 테스트

In [ ]:
from voicesecure.evaluators.adapters.cosyvoice import CosyVoiceAdapter
import numpy as np, gc, torch

cosy = CosyVoiceAdapter(
    model_dir=COSY_MODEL_DIR,
    cosyvoice_root=COSY_ROOT,
    device='cuda',
)
print(f'✓ CosyVoiceAdapter 로드 — sample_rate={cosy._sample_rate}')

# 클론 동작 테스트 (10초 정도)
test_audio = np.random.randn(16000 * 3).astype(np.float32) * 0.05  # 3초 더미
test_audio = np.clip(test_audio, -1.0, 1.0)
cloned = cosy.clone(test_audio, text='안녕하세요 테스트입니다')
print(f'✓ clone 동작 OK — shape={cloned.shape}, dtype={cloned.dtype}')

# 임베딩 추출 테스트
emb = cosy.extract_embedding(test_audio)
print(f'✓ extract_embedding OK — shape={emb.shape}, dtype={emb.dtype}')

del cosy  # 학습 시 train.py 가 새로 로드하므로 여기선 해제
gc.collect(); torch.cuda.empty_cache()

## 셀 6 — KSS 데이터 변환 (폴더당 500 wav)

In [ ]:
import shutil
from pathlib import Path

DATA_DIR = f'{KAGGLE_WORKING}/data_kss_format'

# KSS 자동 탐지 — Kaggle dataset slug 에 따라 경로가 다를 수 있어 자동 탐색
def find_kss():
    """화자 폴더 (숫자명) + wav 가 있는 디렉토리 후보 반환."""
    candidates = set()
    for d in Path('/kaggle/input').rglob('1'):
        if d.is_dir() and d.name.isdigit() and any(d.glob('*.wav')):
            candidates.add(d.parent)
    return sorted(candidates)

kss_candidates = find_kss()
assert kss_candidates, '⚠️ KSS 데이터셋 못 찾음 — 사이드바 Add Data 로 bryanpark/korean-single-speaker-speech-dataset 추가'
KSS_PATH = kss_candidates[0]
print(f'KSS 경로: {KSS_PATH}')

# transcript 자동 탐지
transcript_candidates = list(Path('/kaggle/input').rglob('transcript*.txt'))
transcript_path = transcript_candidates[0] if transcript_candidates else None
print(f'transcript: {transcript_path}')

# 변환
if Path(DATA_DIR).exists():
    print(f'\n기존 {DATA_DIR} 삭제 (이전 잔재 정리)')
    shutil.rmtree(DATA_DIR)

DST = Path(DATA_DIR); DST.mkdir()

labels = {}
if transcript_path:
    for line in transcript_path.read_text(encoding='utf-8').splitlines():
        parts = line.split('|')
        if len(parts) >= 2:
            labels[parts[0].split('/')[-1].replace('.wav','')] = parts[1]
print(f'\n✓ transcript: {len(labels)}개 라벨')

total = 0
for spk in sorted(KSS_PATH.iterdir()):
    if not spk.is_dir() or not spk.name.isdigit(): continue
    dst = DST / spk.name
    dst.mkdir()
    for w in sorted(spk.glob('*.wav'))[:500]:   # 폴더당 500
        shutil.copy(w, dst / w.name)
        total += 1
    print(f'  화자 {spk.name}: {len(list(dst.glob("*.wav")))}개')

label_lines = [f'{w.stem} {labels.get(w.stem, "안녕하세요")}'
               for d in sorted(DST.iterdir()) if d.is_dir()
               for w in sorted(d.glob('*.wav'))]
(DST / 'Labels.txt').write_text('\n'.join(label_lines), encoding='utf-8')

print(f'\n✓ 총 {total} wav 변환 완료')
!ls {DATA_DIR}/

## 셀 7 — 학습 실행 (CosyVoice3)

**🚨 이 셀 실행 전 또는 직후 `Save Version → Save & Run All (Commit)` 누르세요!**

- 약 4~5시간 소요 (2000 wav × 5 epochs)
- TTS 평가 주기 30 (CosyVoice3 가 XTTS 보다 무거움)
- 학습 끝나자마자 자동 tar 백업

In [ ]:
import time, shutil

CHECKPOINT_DIR = f'{KAGGLE_WORKING}/checkpoints'
if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)

t0 = time.time()
!python -u {SDK_ROOT}/train.py \
    --data_dir       {DATA_DIR} \
    --data_format    kss \
    --epochs         5 \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --tts_model_type cosyvoice \
    --cosy_model_dir {COSY_MODEL_DIR} \
    --cosy_root      {COSY_ROOT} \
    --tts_eval_interval 30

print(f'\n총 학습 시간: {(time.time()-t0)/60:.1f}분')

# 즉시 백업 — commit 안 해도 Output 에 들어감
if os.path.exists(CHECKPOINT_DIR):
    !cd {KAGGLE_WORKING} && tar czf checkpoints_backup.tar.gz checkpoints/
    print('✓ 백업: /kaggle/working/checkpoints_backup.tar.gz')
    !ls -la {CHECKPOINT_DIR}/
else:
    print('⚠️ CRITICAL: 학습 실패')

## 셀 8 — 평가 + 청취 + 3-인코더 Transferability

학습된 best.pt 로 변조 → CosyVoice3 클론 → 3개 인코더로 화자 거리 측정.
**핵심 지표:** 원본 vs CosyVoice 클론 거리가 클수록 방어 성공.

In [ ]:
import os, sys, glob, numpy as np, soundfile as sf, torch
from math import gcd
from pathlib import Path
from scipy.signal import resample_poly
from IPython.display import Audio, display

CHECKPOINT_DIR = f'{KAGGLE_WORKING}/checkpoints'

from voicesecure.rl.agent import RLAgent
from voicesecure.modulation.masker import PsychoacousticMasker
from voicesecure.modulation.mixer import Mixer
from voicesecure.evaluators.adapters.wavlm_sv import WavLMSVAdapter
from voicesecure.evaluators.adapters.ecapa_tdnn import ECAPATDNNAdapter
from voicesecure.evaluators.adapters.cosyvoice import CosyVoiceAdapter
from voicesecure.evaluators.base import cosine_distance

SR = 16000

def load_audio(path, sr=SR):
    data, src_sr = sf.read(str(path), dtype='float32', always_2d=True)
    audio = data.mean(axis=1).astype(np.float32)
    if src_sr != sr:
        g = gcd(sr, src_sr)
        audio = resample_poly(audio, sr // g, src_sr // g).astype(np.float32)
    return np.clip(audio, -1.0, 1.0).astype(np.float32)

# 체크포인트 찾기
def find_ckpt():
    p = f'{CHECKPOINT_DIR}/best.pt'
    if os.path.exists(p): return p
    eps = sorted(glob.glob(f'{CHECKPOINT_DIR}/episode_*.pt'),
                 key=lambda p: int(p.rsplit('_',1)[1].split('.')[0]))
    return eps[-1] if eps else None

ckpt = find_ckpt()
assert ckpt, 'checkpoint 없음 — 학습 실패한 듯'
print(f'[ckpt] {ckpt}')

# 어댑터 로드
print('어댑터 로드 중...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
wavlm = WavLMSVAdapter(device=device);   print('  ✓ WavLM')
ecapa = ECAPATDNNAdapter(device=device); print('  ✓ ECAPA')
cosy  = CosyVoiceAdapter(model_dir=COSY_MODEL_DIR, cosyvoice_root=COSY_ROOT, device=device)
print('  ✓ CosyVoice3')

masker = PsychoacousticMasker()
mixer  = Mixer()
agent  = RLAgent()
agent.load(ckpt)
print('✓ best.pt 로드')

# 테스트 샘플
labels = {}
with open(f'{KAGGLE_WORKING}/data_kss_format/Labels.txt', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split(maxsplit=1)
        if len(parts) == 2:
            labels[parts[0]] = parts[1]

wavs = sorted(Path(f'{KAGGLE_WORKING}/data_kss_format').rglob('*.wav'))
test_wav = wavs[100]   # 다른 샘플 보고 싶으면 인덱스 변경
test_text = labels.get(test_wav.stem, '안녕하세요')
print(f'\n[file] {test_wav.name}')
print(f'[text] {test_text}')

# 변조 + 클론
original = load_audio(test_wav)
_, action, _, _, _, _ = agent.act(original, deterministic=False)
safe = masker.clamp(original, action)
modified = mixer.mix(original, safe)
cloned = cosy.clone(modified, text=test_text)

# 3-인코더 transferability 측정
orig_w = wavlm.extract_embedding(original); mod_w = wavlm.extract_embedding(modified); clo_w = wavlm.extract_embedding(cloned)
orig_e = ecapa.extract_embedding(original); mod_e = ecapa.extract_embedding(modified); clo_e = ecapa.extract_embedding(cloned)
orig_c = cosy.extract_embedding(original);  mod_c = cosy.extract_embedding(modified);  clo_c = cosy.extract_embedding(cloned)

print()
print('━━━━━━ 3-인코더 Transferability (cosine dist, 클수록 화자 다름) ━━━━━━')
print(f'{"":<25} {"WavLM":>10} {"ECAPA":>10} {"CosyCAM++":>10}')
print(f'{"원본 vs 변조":<25} {cosine_distance(orig_w, mod_w):>10.4f} {cosine_distance(orig_e, mod_e):>10.4f} {cosine_distance(orig_c, mod_c):>10.4f}')
print(f'{"원본 vs CosyVoice 클론":<25} {cosine_distance(orig_w, clo_w):>10.4f} {cosine_distance(orig_e, clo_e):>10.4f} {cosine_distance(orig_c, clo_c):>10.4f}  ← 핵심')

# 청취
print('\n원본');            display(Audio(original, rate=SR))
print('변조');              display(Audio(modified, rate=SR))
print('CosyVoice 클론');    display(Audio(cloned, rate=SR))

# wav 저장
audio_dir = f'{KAGGLE_WORKING}/audio_samples'
os.makedirs(audio_dir, exist_ok=True)
stem = test_wav.stem
sf.write(f'{audio_dir}/{stem}_original.wav', original, SR)
sf.write(f'{audio_dir}/{stem}_modified.wav', modified, SR)
sf.write(f'{audio_dir}/{stem}_cosyvoice_clone.wav', cloned, SR)
print(f'\n[saved] {audio_dir}/{stem}_*.wav')

## 셀 9 (선택) — 30 샘플 통계 (Ablation: 원본 vs 랜덤 vs 우리 방법)

발표용 통계 결과. 약 10~20분 소요.

In [ ]:
import random

N_SAMPLES = 30
random.seed(42)
sample_idx = random.sample(range(len(wavs)), N_SAMPLES)

results = {'baseline': [], 'random': [], 'ours': []}

for i, idx in enumerate(sample_idx):
    wav_path = wavs[idx]
    text = labels.get(wav_path.stem, '안녕하세요')
    orig = load_audio(wav_path)
    
    # (a) baseline — 원본 직접 클론
    clone_a = cosy.clone(orig, text=text)
    
    # (b) random — 랜덤 노이즈 추가 후 클론
    noise = np.random.randn(len(orig)).astype(np.float32) * 0.02
    rand_modified = np.clip(orig + noise, -1.0, 1.0)
    clone_b = cosy.clone(rand_modified, text=text)
    
    # (c) ours — 학습된 정책 변조 후 클론
    _, action, _, _, _, _ = agent.act(orig, deterministic=False)
    safe = masker.clamp(orig, action)
    our_modified = mixer.mix(orig, safe)
    clone_c = cosy.clone(our_modified, text=text)
    
    # WavLM 으로 측정 (학습에 사용된 인코더 — transferability 측정의 baseline)
    orig_emb = wavlm.extract_embedding(orig)
    d_a = cosine_distance(orig_emb, wavlm.extract_embedding(clone_a))
    d_b = cosine_distance(orig_emb, wavlm.extract_embedding(clone_b))
    d_c = cosine_distance(orig_emb, wavlm.extract_embedding(clone_c))
    
    results['baseline'].append(d_a)
    results['random'].append(d_b)
    results['ours'].append(d_c)
    
    print(f'[{i+1:2d}/{N_SAMPLES}] {wav_path.name}: baseline={d_a:.4f}, random={d_b:.4f}, ours={d_c:.4f}')

print()
print('━━━━━━ Ablation 통계 (WavLM cosine dist, 클수록 방어 성공) ━━━━━━')
for k, v in results.items():
    arr = np.array(v)
    print(f'  {k:>10}: mean={arr.mean():.4f}, std={arr.std():.4f}, min={arr.min():.4f}, max={arr.max():.4f}')

gain = np.array(results['ours']).mean() - np.array(results['baseline']).mean()
gain_vs_rand = np.array(results['ours']).mean() - np.array(results['random']).mean()
print(f'\n  ours - baseline = {gain:+.4f}  (학습 효과)')
print(f'  ours - random   = {gain_vs_rand:+.4f}  (단순 노이즈 대비 우위)')